# causal_direction_inference: plots (v2, new pipeline)

Reads the CSVs produced by `run_causal_direction_v2.py` /
`aggregate_causal_direction_results.py` (Part A), `run_cascade_v2.py` /
`score_cascade_results.py` (Parts B/C), and `run_grnboost2_gmm.py` (Part D
-- GMM threshold only, no shuffle-null generator).

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

from twinfer.utils.paths import stage_dir, get_repo_root

# ============================================================
# Fonts / style (matches plot.ipynb)
# ============================================================
plt.rcParams['pdf.fonttype'] = 42  # For PDF export
plt.rcParams['ps.fonttype'] = 42   # For PostScript (EPS) export
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 18*1.5      # x/y labels
plt.rcParams['axes.titlesize'] = 20*1.5
plt.rcParams['xtick.labelsize'] = 12*1.5     # x-axis tick labels
plt.rcParams['ytick.labelsize'] = 12*1.5    # y-axis tick labels
plt.rcParams['legend.fontsize'] = 12*1.5     # legend text

path_to_code_repo = get_repo_root()
import matplotlib.font_manager as fm
# ============================================================
# Fonts / style
# ============================================================
font_paths = [
    f"{path_to_code_repo}/package/twinfer/plotting/assets/fonts/Arial.ttf",
     f"{path_to_code_repo}/package/twinfer/plotting/assets/fonts/Arial Bold.ttf",
     f"{path_to_code_repo}/package/twinfer/plotting/assets/fonts/Arial Italic.ttf",
     f"{path_to_code_repo}/package/twinfer/plotting/assets/fonts/Arial Bold Italic.ttf",
]

for fp in font_paths:
    try:
        fm.fontManager.addfont(fp)
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

# ==== LaTeX + SVG text mode (Illustrator-safe) ====
# ==== LaTeX + SVG text mode (Illustrator-safe) ====
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42  # For PDF export
plt.rcParams['ps.fonttype'] = 42   # For PostScript (EPS) export
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 18*1.5      # x/y labels
plt.rcParams['axes.titlesize'] = 20*1.5 
plt.rcParams['xtick.labelsize'] = 12*1.5     # x-axis tick labels
plt.rcParams['ytick.labelsize'] = 12*1.5    # y-axis tick labels
plt.rcParams['legend.fontsize'] = 12*1.5     # legend text


def make_reds_blues_colormap(vmin=-0.05, vmax=0.18):
    """Custom red-white-blue colormap with pure white at 0, asymmetric. Verbatim from plot.ipynb."""
    zero_position = max(0, (0 - vmin) / (vmax - vmin))
    n_total = 256
    n_reds = int(zero_position * n_total)
    n_blues = n_total - n_reds
    red_intensity = abs(vmin) / max(abs(vmin), abs(vmax))
    blue_intensity = abs(vmax) / max(abs(vmin), abs(vmax))
    reds = plt.cm.Reds(np.linspace(0.8 * red_intensity, 0, n_reds))
    whites = np.ones((1, 4))
    blues = plt.cm.Blues(np.linspace(0, 0.8 * blue_intensity, n_blues))
    colors = np.vstack((reds, whites, blues))
    return LinearSegmentedColormap.from_list('RedsBlues', colors)


path_to_plot_data = str(stage_dir("causal_direction_inference", "analysis", run_tag="latest"))
path_to_cascade_results = f'{TWINFER_PROJECT_ROOT}/analysis_data/paper_analysis/causal_direction_inference/slurm_run/cascade_results'
path_to_grnboost_results = f'{TWINFER_PROJECT_ROOT}/analysis_data/paper_analysis/causal_direction_inference/grnboost2_results'
path_to_plots = f'{TWINFER_PROJECT_ROOT}/analysis_data/paper_analysis/causal_direction_inference/plots'
os.makedirs(path_to_plots, exist_ok=True)
print("reading analysis results from:", path_to_plot_data)

## Panel: 2-gene direction cross-correlations vs. thresholds, 6 conditions (Part A)

In [ ]:
def create_box_plot_and_save(df_results, save_dir):
    """
    Plot 1: Correlations colored by threshold (blue > threshold, red < threshold, grey in between)
    Plot 2: Threshold boxplots in grey
    Verbatim from the original plot.ipynb -- schema-compatible with the new
    box_plot_data.csv (Condition, gene_1_to_gene_2, gene_2_to_gene_1,
    threshold_12, threshold_21, threshold_combined).
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    fig.patch.set_facecolor('none')
    ax.set_facecolor('none')

    conditions = df_results['Condition'].unique()

    all_values_12, all_values_21 = [], []
    all_thresholds, all_thresholds_12, all_thresholds_21 = [], [], []

    for condition in conditions:
        condition_data = df_results[df_results['Condition'] == condition]
        all_values_12.append(condition_data['gene_1_to_gene_2'].values.tolist())
        all_values_21.append(condition_data['gene_2_to_gene_1'].values.tolist())
        all_thresholds_12.append(condition_data['threshold_12'].values.tolist())
        all_thresholds_21.append(condition_data['threshold_21'].values.tolist())
        all_thresholds.append(condition_data['threshold_combined'].values.tolist())

    flat_thresholds = [item for sublist in all_thresholds for item in sublist]

    ax.axhline(np.mean(flat_thresholds), linestyle="--", color="black", linewidth=1, alpha=0.7, dashes=(12, 4))
    ax.axhline(-np.mean(flat_thresholds), linestyle="--", color="black", linewidth=1, alpha=0.7, dashes=(12, 4))

    def get_threshold_color(values, thresholds):
        median_val = np.median(values)
        mean_threshold = np.mean(thresholds)
        if median_val > mean_threshold:
            return 'blue'
        elif median_val < -mean_threshold:
            return 'red'
        return 'grey'

    positions_12 = [i - 0.2 for i in range(1, len(conditions) + 1)]
    box_12 = ax.boxplot(all_values_12, positions=positions_12, patch_artist=True, showfliers=False, widths=0.3)
    positions_21 = [i + 0.2 for i in range(1, len(conditions) + 1)]
    box_21 = ax.boxplot(all_values_21, positions=positions_21, patch_artist=True, showfliers=False, widths=0.3)

    solid = mlines.Line2D([], [], color="black", linestyle="-", label=r"$\hat{\rho}^{\dagger}_{x(t_{1}) \to y(t_{2})}$")
    dashed = mlines.Line2D([], [], color="black", linestyle="--", label=r"$\hat{\rho}^{\dagger}_{y(t_{1}) \to x(t_{2})}$")
    ax.legend(handles=[solid, dashed])

    for patch, values, thresholds in zip(box_12['boxes'], all_values_12, all_thresholds_12):
        color = get_threshold_color(values, thresholds)
        patch.set_facecolor('none'); patch.set_edgecolor(color); patch.set_linewidth(2); patch.set_clip_on(False)
    for median, values, thresholds in zip(box_12['medians'], all_values_12, all_thresholds_12):
        median.set_color(get_threshold_color(values, thresholds)); median.set_linewidth(2); median.set_clip_on(False)

    for patch, values, thresholds in zip(box_21['boxes'], all_values_21, all_thresholds_21):
        color = get_threshold_color(values, thresholds)
        patch.set_facecolor('none'); patch.set_edgecolor(color); patch.set_linestyle("--"); patch.set_linewidth(2); patch.set_clip_on(False)
    for median, values, thresholds in zip(box_21['medians'], all_values_21, all_thresholds_21):
        median.set_color(get_threshold_color(values, thresholds)); median.set_linewidth(2); median.set_clip_on(False)

    for box, all_values, all_thresh, is_dashed in [(box_12, all_values_12, all_thresholds_12, False), (box_21, all_values_21, all_thresholds_21, True)]:
        for i, whisker in enumerate(box['whiskers']):
            idx = i // 2
            color = get_threshold_color(all_values[idx], all_thresh[idx])
            whisker.set_color(color); whisker.set_linewidth(2); whisker.set_clip_on(False)
        for i, cap in enumerate(box['caps']):
            idx = i // 2
            color = get_threshold_color(all_values[idx], all_thresh[idx])
            cap.set_color(color); cap.set_linewidth(2); cap.set_clip_on(False)

    ax.set_xticks(range(1, len(conditions) + 1))
    ax.set_xticklabels(list(conditions), rotation=30, ha="right")
    ax.tick_params(axis='both', which='both', bottom=True, top=False, left=True, right=False, length=4, width=1)
    plt.ylabel(r"twin cross-correlation")
    plt.savefig(os.path.join(save_dir, 'correlations_with_thresholds.svg'), format='svg', bbox_inches='tight', facecolor='none', edgecolor='none', transparent=True)
    plt.savefig(os.path.join(save_dir, 'correlations_with_thresholds.pdf'), format='pdf', bbox_inches='tight', facecolor='none', edgecolor='none', transparent=True)
    plt.show()
    print(f"Saved plots to {save_dir}")


df_results = pd.read_csv(f"{path_to_plot_data}/box_plot_data.csv")
create_box_plot_and_save(df_results, path_to_plots)

## Panel: rescued pairs (Part 0 -- opposite-sign cross-correlation rescue)

In [ ]:
rescued_df = pd.read_csv(f"{path_to_plot_data}/rescued_from_no_regulation.csv")
print(f"{len(rescued_df)} replicate-pairs qualified for the opposite-sign rescue screen")
print(rescued_df['condition'].value_counts())

## Panel: 5-gene cascade -- observed correlations and inferred edges (Part B)

In [ ]:
direction_matrix

In [ ]:
real

In [ ]:
directed_edges

In [ ]:
cascade_json = sorted(glob.glob(f"{path_to_cascade_results}/cascade_rep_*.json"))[0]
with open(cascade_json) as f:
    cascade = json.load(f)

real = cascade["real"]
genes = real["genes"]
gene_labels = [f"g{g.split('_')[1]}" for g in genes]

gene_gene_corr = pd.DataFrame(real["pairwise_gene_gene_correlation_matrix"]).loc[genes, genes]
direction_matrix = pd.DataFrame(real["unfiltered_direction_matrix"]).loc[genes, genes]
directed_edges = set(tuple(e) for e in real["directed_edges"])
classification = real["classification"]

# Candidates tested for direction (Step 4 input): non-self pairs that passed
# Step 1-3, plus the opposite-sign rescue screen. Matches plot.ipynb's
# "potential_regulation" -- the set that gets a black outline in the
# gene-gene correlation panel only.
potential_regulation = [
    tuple(p) for p in (
        classification["single_state_regulation"]
        + classification["multiple_states_no_reg"]  
        + classification["multiple_states_and_reg"]
        # + real["rescued_from_no_regulation"]
    )
    if p[0] != p[1]
]
final_edges_no_self = [tuple(e) for e in real["directed_edges"] if e[0] != e[1]]
ambiguous_pairs = [tuple(p) for p in (classification["multiple_states_and_reg"] + classification["multiple_states_no_reg"]) if p[0] != p[1]]


def plot_cascade_heatmap(matrix_df, keep_pairs, cbar_label, out_name,
                          outline_pairs=None, dashed_diagonal_pairs=None, symmetric=True):
    """Gridspec heatmap: white=0 for non-kept cells, grey diagonal, red-blue
    colormap -- matches plot.ipynb's gene_correlation / direction_correlation
    heatmap style. outline_pairs draws a black rectangle around kept cells
    (only used for the gene-gene correlation panel, like plot.ipynb cell 10);
    dashed_diagonal_pairs draws a dashed diagonal for ambiguous multi-state
    pairs (like plot.ipynb cell 11)."""
    gene_to_idx = {g: i for i, g in enumerate(matrix_df.index)}
    data_matrix = matrix_df.to_numpy(float)

    # White (0) everywhere except kept pairs; grey (NaN) only on the diagonal.
    plot_matrix = np.zeros_like(data_matrix)
    for a, b in keep_pairs:
        if a in gene_to_idx and b in gene_to_idx:
            i, j = gene_to_idx[a], gene_to_idx[b]
            plot_matrix[i, j] = data_matrix[i, j]
            if symmetric:
                plot_matrix[j, i] = data_matrix[j, i]
    np.fill_diagonal(plot_matrix, np.nan)

    max_abs = np.nanmax(np.abs(plot_matrix))
    max_abs = max_abs if np.isfinite(max_abs) and max_abs > 0 else 0.05
    vmin, vmax = -max_abs, max_abs
    cmap = make_reds_blues_colormap(vmin=vmin, vmax=vmax)
    cmap.set_bad(color="#D9D9D9")

    fig = plt.figure(figsize=(6, 6))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.05, 0.95], hspace=0.1)
    cbar_ax = fig.add_subplot(gs[0])
    heatmap_ax = fig.add_subplot(gs[1])

    sns.heatmap(
        pd.DataFrame(plot_matrix, index=matrix_df.index, columns=matrix_df.columns),
        ax=heatmap_ax, cmap=cmap, vmin=vmin, vmax=vmax, square=True,
        cbar=True, cbar_ax=cbar_ax, cbar_kws={"orientation": "horizontal"},
        linewidths=0.5, linecolor="black",
    )

    if outline_pairs:
        for a, b in outline_pairs:
            if a in gene_to_idx and b in gene_to_idx:
                i, j = gene_to_idx[a], gene_to_idx[b]
                for (x, y) in [(j, i), (i, j)]:
                    rect = Rectangle((x, y), 1, 1, fill=False, edgecolor="black",
                                      linewidth=1.5, clip_on=False)
                    heatmap_ax.add_patch(rect)

    if dashed_diagonal_pairs:
        for a, b in dashed_diagonal_pairs:
            if a in gene_to_idx and b in gene_to_idx:
                i, j = gene_to_idx[a], gene_to_idx[b]
                heatmap_ax.plot([j, j + 1], [i, i + 1], linestyle="--", color="black",
                                 linewidth=1.5, clip_on=False)
                heatmap_ax.plot([i, i + 1], [j, j + 1], linestyle="--", color="black",
                                 linewidth=1.5, clip_on=False)

    cbar_ax.xaxis.set_label_position("top")
    cbar_ax.set_xlabel(cbar_label, labelpad=10)
    cbar_ax.xaxis.tick_top()

    heatmap_ax.set_xticklabels(gene_labels, rotation=90, ha="center", va="top")
    heatmap_ax.set_yticklabels(gene_labels, rotation=0, ha="right", va="center")

    # --- transparent background ---
    fig.patch.set_alpha(0)
    for ax in [heatmap_ax, cbar_ax]:
        ax.set_facecolor("none")
    for im in heatmap_ax.get_images() + cbar_ax.get_images():
        im.set_facecolor((1, 1, 1, 0))
        im.set_edgecolor((1, 1, 1, 0))
    for spine in heatmap_ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_edgecolor("black")
        spine.set_clip_on(False)

    # --- save ---
    out = Path(path_to_plots)
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{out_name}.png", dpi=300, bbox_inches="tight", transparent=True)
    fig.savefig(out / f"{out_name}.svg", bbox_inches="tight", transparent=True,
                facecolor="none", edgecolor="none", format="svg")
    fig.savefig(out / f"{out_name}.pdf", bbox_inches="tight", transparent=True,
                facecolor="none", edgecolor="none", format="pdf")
    print(f"Saved PNG/SVG/PDF: {out / out_name}.*")
    plt.show()


# Step 1: gene-gene correlation -- only direction candidates keep their
# value, and only this panel gets the black outline (matches plot.ipynb).
plot_cascade_heatmap(
    gene_gene_corr, potential_regulation,
    r"gene correlation $\rho$", "cascade_gene_correlation",
    outline_pairs=potential_regulation,
)

# Step 4: significant directed edges only -- no outline, dashed diagonal for
# ambiguous multi-state pairs (matches plot.ipynb).
plot_cascade_heatmap(
    direction_matrix, final_edges_no_self,
    r"cross-correlation $\rho^{\dagger}$", "cascade_direction_correlation",
    dashed_diagonal_pairs=ambiguous_pairs, symmetric=False
)

print("Directed edges (real network):", sorted(directed_edges))
print("Rescued from no_regulation:", real["rescued_from_no_regulation"])
print()
random_ = cascade["random"]
print("Directed edges (remove_twin_structure control):", random_["directed_edges"])

## Panel: TwINFER vs. GRNBoost2 (GMM threshold) vs. random baseline (Parts C/D)

In [ ]:
metrics_twinfer = pd.read_csv(f"{path_to_cascade_results}/metrics_TwINFER.csv")

grnboost_jsons = sorted(glob.glob(f"{path_to_grnboost_results}/grnboost2_rep_*.json"))
grnboost_rows = []
for p in grnboost_jsons:
    with open(p) as f:
        d = json.load(f)
    grnboost_rows.append({
        "rep_id": d["rep_id"], "precision_gmm": d["precision"], "recall_gmm": d["recall"], "f1_gmm": d["f1"],
        "precision_random": d["precision_random"], "recall_random": d["recall_random"], "f1_random": d["f1_random"],
    })
metrics_grnboost = pd.DataFrame(grnboost_rows)

metrics_to_plot = ["precision", "recall", "f1"]


def get_mean_std(df, cols):
    return df[cols].mean().values, df[cols].std().values


means_rand, stds_rand = get_mean_std(metrics_grnboost, ["precision_random", "recall_random", "f1_random"])
means_gmm, stds_gmm = get_mean_std(metrics_grnboost, ["precision_gmm", "recall_gmm", "f1_gmm"])
means_twin, stds_twin = get_mean_std(metrics_twinfer, metrics_to_plot)

n_methods = 3
x = np.arange(len(metrics_to_plot))
width = 0.2
offsets = np.linspace(-(n_methods - 1) / 2, (n_methods - 1) / 2, n_methods) * width

fig, ax = plt.subplots(figsize=(8, 5))

bar_specs = [
    (offsets[0], means_rand, stds_rand, "Random", "0.85"),
    (offsets[1], means_gmm, stds_gmm, "GRNBoost2 (GMM)", "0.55"),
    (offsets[2], means_twin, stds_twin, "TwINFER", "#2c7bb6"),
]

all_bars = []
for offset, means, stds, label, color in bar_specs:
    bars = ax.bar(x + offset, means, width, yerr=stds, capsize=4, label=label, color=color,
                   error_kw={"elinewidth": 0.8})
    all_bars.append(bars)

# Disable clipping
for container in all_bars:
    for patch in container.patches:
        patch.set_clip_on(False)
for line in ax.lines:
    line.set_clip_on(False)
for spine in ax.spines.values():
    spine.set_clip_on(False)
ax.patch.set_clip_on(False)
fig.patch.set_clip_on(False)

ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics_to_plot])
ax.set_ylabel("score")
ax.set_ylim(0, 1.05)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

leg = ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
leg.set_clip_on(False)

plt.tight_layout()

# --- save ---
out = Path(path_to_plots)
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "metrics_comparison.svg", bbox_inches="tight", transparent=True)
fig.savefig(out / "metrics_comparison.pdf", bbox_inches="tight", transparent=True)
fig.savefig(out / "metrics_comparison.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()